## import

In [37]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV
import xgboost as xgb

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import joblib

In [2]:
df_clean = pd.read_csv("../data/cleaned_climber_df.csv")

df_clean.head()

,sex,height,weight,age,years_cl,grades_count,grades_first,grades_last,grades_max,grades_mean,...,country_NLD,country_NOR,country_POL,country_PRT,country_RUS,country_SVN,country_SWE,country_USA,country_ZAF,country_other
0,0,177,73,41.0,21,84,36,55,62,46.750000,...,0,0,0,0,0,0,1,0,0,0
1,0,180,78,44.0,22,12,53,51,59,52.833333,...,0,0,0,0,0,0,1,0,0,0
2,1,165,58,33.0,16,119,53,49,64,53.890756,...,0,0,0,0,0,0,1,0,0,0
3,0,167,63,52.0,25,298,53,49,63,49.406040,...,0,0,0,0,0,0,1,0,0,0
4,0,177,68,44.0,21,5,53,49,53,51.400000,...,0,1,0,0,0,0,0,0,0,0


In [6]:
X = df_clean.drop(columns=['grades_max'])
y = df_clean['grades_max']

In [21]:
X = np.array(X)
y = np.array(y)

In [16]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)
print(X_scaled[:5])

[[-0.37814234  0.09966659  0.55713497  1.00994733  1.36338861  0.02974053
  -1.01805407  0.85118829  0.15776785 -2.71829226 -0.15112917 -0.17838855
  -0.12794753 -0.20165978 -0.16973758 -0.14499692 -0.08479153 -0.2201254
  -0.10493028 -0.36843551 -0.07304983 -0.24252387 -0.14598083 -0.12305718
  -0.28068433 -0.11637157 -0.14760731 -0.18324646 -0.28671002 -0.13123307
  -0.15795808 -0.1171768   4.03603233 -0.35908069 -0.17452852 -0.31458559]
 [-0.37814234  0.45226431  1.07383081  1.40517085  1.52710371 -0.47943587
   0.77562254  0.42645415  0.92868879 -2.95860212 -0.15112917 -0.17838855
  -0.12794753 -0.20165978 -0.16973758 -0.14499692 -0.08479153 -0.2201254
  -0.10493028 -0.36843551 -0.07304983 -0.24252387 -0.14598083 -0.12305718
  -0.28068433 -0.11637157 -0.14760731 -0.18324646 -0.28671002 -0.13123307
  -0.15795808 -0.1171768   4.03603233 -0.35908069 -0.17452852 -0.31458559]
 [ 2.6445068  -1.31072429 -0.99295257 -0.04398204  0.54481307  0.27725683
   0.77562254  0.21408708  1.06269254 

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, train_size=0.8 ,random_state=42)

In [22]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((8741, 36), (8741,), (2186, 36), (2186,))

## Modeling

In [32]:
xgb_model = xgb.XGBRegressor(objective='reg:linear', random_state=42)
xgb_model.fit(X_train, y_train)

c:\Users\Tomca\anaconda3\envs\hera_env\Lib\site-packages\xgboost\training.py:199: UserWarning: [17:49:07] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\objective\regression_obj.cu:282: reg:linear is now deprecated in favor of reg:squarederror.
  bst.update(dtrain, iteration=i, fobj=obj)


,objective,'reg:linear'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [33]:
y_pred = xgb_model.predict(X_test)

In [34]:
mae = mean_absolute_error(y_pred, y_test)
mse = mean_squared_error(y_pred, y_test)
rmse = np.sqrt(mse)
r2 = r2_score(y_pred, y_test)

In [35]:
print("MAE : ", mae)
print("RMSE : ", rmse)
print("r2 : ", r2)

MAE :  1.7284655570983887
RMSE :  2.3835237035771044
r2 :  0.9327163100242615


## H tuning

In [46]:
params = {'objective':['reg:squarederror'],
              'booster':['gbtree','gblinear'],
              'learning_rate': [0.1], 
              'max_depth': [7,10,15,20],
              'min_child_weight': [10,15,20,25],
              'colsample_bytree': [0.8, 0.9, 1],
              'n_estimators': [300,400,500,600],
              "reg_alpha"   : [0.5,0.2,1],
              "reg_lambda"  : [2,3,5],
              "gamma"       : [1,2,3]}

In [ ]:
xgb_reg = xgb.XGBRegressor()
xgb_reg = RandomizedSearchCV(xgb_reg, param_distributions=params, n_iter=15, scoring='neg_mean_absolute_error', n_jobs=12, cv=5, verbose=3)

TypeError: GridSearchCV.__init__() got an unexpected keyword argument 'param_distributions'

In [49]:
xgb_reg.fit(X_train, y_train)

Fitting 5 folds for each of 15 candidates, totalling 75 fits


,estimator,"XGBRegressor(...ree=None, ...)"
,param_distributions,"{'booster': ['gbtree', 'gblinear'], 'colsample_bytree': [0.8, 0.9, ...], 'gamma': [1, 2, ...], 'learning_rate': [0.1], ...}"
,n_iter,15
,scoring,'neg_mean_absolute_error'
,n_jobs,12
,refit,True
,cv,5
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,None
,error_score,nan


In [51]:
y_pred = xgb_reg.predict(X_test)

In [52]:
mae = mean_absolute_error(y_pred, y_test)
mse = mean_squared_error(y_pred, y_test)
rmse = np.sqrt(mse)
r2 = r2_score(y_pred, y_test)

In [53]:
print("MAE : ", mae)
print("RMSE : ", rmse)
print("r2 : ", r2)

MAE :  1.6009409427642822
RMSE :  2.259115931852233
r2 :  0.93953937292099


In [55]:
joblib.dump(xgb_reg, "../models/xgb_climbing.joblib")

['../models/xgb_climbing.joblib']